In [ ]:
import sys
sys.path = [p for p in sys.path if 'Neuron and Synapse Models' not in p and 'Tools' not in p]
sys.path.append('Neuron and Synapse Models')
sys.path.append('Tools')


import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, Layout, interactive_output, FloatSlider, IntSlider, Dropdown, HTML, FloatText, Label, ToggleButton


#for dynapse
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1
import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
import params_all_cores
from params_all_cores import set_params
import time
import importlib

from IPython.display import display, clear_output
from ipywidgets import widgets
from ipywidgets import IntSlider, FloatSlider, Dropdown, HTML, FloatText, Label, ToggleButton 
from ipywidgets import VBox, HBox, Layout, Tab
import numpy as np
from math import pi
from ipywidgets import Output
import threading
import time
import matplotlib.pyplot as plt

from collections import deque

In [ ]:
global sink_node
sink_node = 0
global NBINS
global ring_pops

In [ ]:
eventsBuffer = deque(maxlen=500)

# to be used when connecting to Dynap-se locally 
devices = samna.device.get_unopened_devices()
model   = samna.device.open_device(devices[int(0)])

In [ ]:
def collect_spikes(sink_node):
    eventsBuffer.extend(sink_node.get_events())#dynapse
api = model.get_dynapse1_api()

In [ ]:
slider_width='300px'
description_width='80px'

# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=1,
    max=15, 
    step=1, 
    value=10, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.1, 
    value=np.pi, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.1, 
    value=0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

velocity_input_slider = FloatSlider(
    min=-2.0, 
    max=2.0, 
    step=0.1, 
    value=0.0, 
    description='Velocity Input (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

# Input boxes for durations
duration_box = FloatText(
    value=2,
    description='Simulation Duration (s):',
    style={'description_width': '150px'}
)

input_duration_box = FloatText(
    value=0.5,
    description='Input Duration (s):',
    style={'description_width': '150px'}
)

autapse_button = ToggleButton(
    value=False,
    description='Autapse',
    tooltip='Allows autapse connections',
    button_style=''
)

# Create sliders for Dynapse parameters with correct ranges
# Ring population parameters (core 3)
# AMPA weight
PS_WEIGHT_EXC_F_N_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='AMPA Weight (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_EXC_F_N_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=20, 
    description='AMPA Weight (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# NMDA weight
PS_WEIGHT_EXC_S_N_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='NMDA Weight (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_EXC_S_N_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=40, 
    description='NMDA Weight (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# GABA B weight
PS_WEIGHT_INH_S_N_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='GABA B Weight (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_INH_S_N_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=120, 
    description='GABA B Weight (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# AMPA gain
NPDPIE_THR_F_P_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='AMPA Gain (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPIE_THR_F_P_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='AMPA Gain (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# NMDA gain
NPDPIE_THR_S_P_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='NMDA Gain (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPIE_THR_S_P_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='NMDA Gain (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# GABA B gain
NPDPII_THR_S_P_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='GABA B Gain (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPII_THR_S_P_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='GABA B Gain (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# Inhibitory population parameters (core 1)
# NMDA weight
PS_WEIGHT_EXC_S_N_INHPOP_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='NMDA Weight Inh (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_EXC_S_N_INHPOP_fine_slider = IntSlider(
    min=0, max=255, step=1, value=30, 
    description='NMDA Weight Inh (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# NMDA gain
NPDPIE_THR_S_P_INHPOP_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='NMDA Gain Inh (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPIE_THR_S_P_INHPOP_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='NMDA Gain Inh (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# Create a toggle button for hardware implementation
run_dynapse_button = ToggleButton(
    value=False,
    description='Run on Dynapse',
    button_style='warning',
    tooltip='Click to run on Dynapse hardware',
    icon='microchip'
)

# Add required input parameters specific to the Brian2 simulation
I0_slider = FloatSlider(
    min=1, 
    max=50, 
    step=1, 
    value=1, 
    description='Input Current I0 Amplitude (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
)


widgets = {
    'num_neurons_slider': num_neurons_slider,
    'stimulus_center_slider': stimulus_center_slider,
    'stimulus_width_slider': stimulus_width_slider,
    'velocity_input_slider': velocity_input_slider,
    'duration_box': duration_box,
    'input_duration_box': input_duration_box,
    'autapse_button': autapse_button,
    'PS_WEIGHT_EXC_F_N_RING_coarse_slider': PS_WEIGHT_EXC_F_N_RING_coarse_slider,
    'PS_WEIGHT_EXC_F_N_RING_fine_slider': PS_WEIGHT_EXC_F_N_RING_fine_slider,
    'PS_WEIGHT_EXC_S_N_RING_coarse_slider': PS_WEIGHT_EXC_S_N_RING_coarse_slider,
    'PS_WEIGHT_EXC_S_N_RING_fine_slider': PS_WEIGHT_EXC_S_N_RING_fine_slider,
    'PS_WEIGHT_INH_S_N_RING_coarse_slider': PS_WEIGHT_INH_S_N_RING_coarse_slider,
    'PS_WEIGHT_INH_S_N_RING_fine_slider': PS_WEIGHT_INH_S_N_RING_fine_slider,
    'NPDPIE_THR_F_P_RING_coarse_slider': NPDPIE_THR_F_P_RING_coarse_slider,
    'NPDPIE_THR_F_P_RING_fine_slider': NPDPIE_THR_F_P_RING_fine_slider,
    'NPDPIE_THR_S_P_RING_coarse_slider': NPDPIE_THR_S_P_RING_coarse_slider,
    'NPDPIE_THR_S_P_RING_fine_slider': NPDPIE_THR_S_P_RING_fine_slider,
    'NPDPII_THR_S_P_RING_coarse_slider': NPDPII_THR_S_P_RING_coarse_slider,
    'NPDPII_THR_S_P_RING_fine_slider': NPDPII_THR_S_P_RING_fine_slider,
    'PS_WEIGHT_EXC_S_N_INHPOP_coarse_slider': PS_WEIGHT_EXC_S_N_INHPOP_coarse_slider,
    'PS_WEIGHT_EXC_S_N_INHPOP_fine_slider': PS_WEIGHT_EXC_S_N_INHPOP_fine_slider,
    'NPDPIE_THR_S_P_INHPOP_coarse_slider': NPDPIE_THR_S_P_INHPOP_coarse_slider,
    'NPDPIE_THR_S_P_INHPOP_fine_slider': NPDPIE_THR_S_P_INHPOP_fine_slider,
    'I0_slider': I0_slider,
    'run_dynapse_button': run_dynapse_button
}


In [ ]:
slider_width='300px'
description_width='80px'

# Create sliders for parameters
num_neurons_slider = IntSlider(
    min=1,
    max=15, 
    step=1, 
    value=10, 
    description='Number of Neurons:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_center_slider = FloatSlider(
    min=0, 
    max=2*pi, 
    step=0.1, 
    value=np.pi, 
    description='Stimulus Center (rad):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

stimulus_width_slider = FloatSlider(
    min=0.1, 
    max=2.0, 
    step=0.1, 
    value=0.5, 
    description='Stimulus Width:', 
    continuous_update=False,
    style={'description_width': '150px'},
)

velocity_input_slider = FloatSlider(
    min=-2.0, 
    max=2.0, 
    step=0.1, 
    value=0.0, 
    description='Velocity Input (mV):', 
    continuous_update=False,
    style={'description_width': '150px'},
)

# Input boxes for durations
duration_box = FloatText(
    value=2,
    description='Simulation Duration (s):',
    style={'description_width': '150px'}
)

input_duration_box = FloatText(
    value=0.5,
    description='Input Duration (s):',
    style={'description_width': '150px'}
)

autapse_button = ToggleButton(
    value=False,
    description='Autapse',
    tooltip='Allows autapse connections',
    button_style=''
)

# Create sliders for Dynapse parameters with correct ranges
# Ring population parameters (core 3)
# AMPA weight
PS_WEIGHT_EXC_F_N_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='AMPA Weight (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_EXC_F_N_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=20, 
    description='AMPA Weight (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# NMDA weight
PS_WEIGHT_EXC_S_N_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='NMDA Weight (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_EXC_S_N_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=40, 
    description='NMDA Weight (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# GABA B weight
PS_WEIGHT_INH_S_N_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='GABA B Weight (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_INH_S_N_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=120, 
    description='GABA B Weight (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# AMPA gain
NPDPIE_THR_F_P_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='AMPA Gain (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPIE_THR_F_P_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='AMPA Gain (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# NMDA gain
NPDPIE_THR_S_P_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='NMDA Gain (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPIE_THR_S_P_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='NMDA Gain (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# GABA B gain
NPDPII_THR_S_P_RING_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='GABA B Gain (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPII_THR_S_P_RING_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='GABA B Gain (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# Inhibitory population parameters (core 1)
# NMDA weight
PS_WEIGHT_EXC_S_N_INHPOP_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=6, 
    description='NMDA Weight Inh (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

PS_WEIGHT_EXC_S_N_INHPOP_fine_slider = IntSlider(
    min=0, max=255, step=1, value=30, 
    description='NMDA Weight Inh (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# NMDA gain
NPDPIE_THR_S_P_INHPOP_coarse_slider = IntSlider(
    min=0, max=7, step=1, value=4, 
    description='NMDA Gain Inh (coarse):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

NPDPIE_THR_S_P_INHPOP_fine_slider = IntSlider(
    min=0, max=255, step=1, value=80, 
    description='NMDA Gain Inh (fine):', 
    continuous_update=False,
    style={'description_width': description_width},
    layout=Layout(width=slider_width),
)

# Create a toggle button for hardware implementation
run_dynapse_button = ToggleButton(
    value=False,
    description='Run on Dynapse',
    button_style='warning',
    tooltip='Click to run on Dynapse hardware',
    icon='microchip'
)

# Add required input parameters specific to the Brian2 simulation
I0_slider = FloatSlider(
    min=1, 
    max=50, 
    step=1, 
    value=1, 
    description='Input Current I0 Amplitude (mV):',
    continuous_update=False,
    style={'description_width': '150px'},
)


widgets = {
    'num_neurons_slider': num_neurons_slider,
    'stimulus_center_slider': stimulus_center_slider,
    'stimulus_width_slider': stimulus_width_slider,
    'velocity_input_slider': velocity_input_slider,
    'duration_box': duration_box,
    'input_duration_box': input_duration_box,
    'autapse_button': autapse_button,
    'PS_WEIGHT_EXC_F_N_RING_coarse_slider': PS_WEIGHT_EXC_F_N_RING_coarse_slider,
    'PS_WEIGHT_EXC_F_N_RING_fine_slider': PS_WEIGHT_EXC_F_N_RING_fine_slider,
    'PS_WEIGHT_EXC_S_N_RING_coarse_slider': PS_WEIGHT_EXC_S_N_RING_coarse_slider,
    'PS_WEIGHT_EXC_S_N_RING_fine_slider': PS_WEIGHT_EXC_S_N_RING_fine_slider,
    'PS_WEIGHT_INH_S_N_RING_coarse_slider': PS_WEIGHT_INH_S_N_RING_coarse_slider,
    'PS_WEIGHT_INH_S_N_RING_fine_slider': PS_WEIGHT_INH_S_N_RING_fine_slider,
    'NPDPIE_THR_F_P_RING_coarse_slider': NPDPIE_THR_F_P_RING_coarse_slider,
    'NPDPIE_THR_F_P_RING_fine_slider': NPDPIE_THR_F_P_RING_fine_slider,
    'NPDPIE_THR_S_P_RING_coarse_slider': NPDPIE_THR_S_P_RING_coarse_slider,
    'NPDPIE_THR_S_P_RING_fine_slider': NPDPIE_THR_S_P_RING_fine_slider,
    'NPDPII_THR_S_P_RING_coarse_slider': NPDPII_THR_S_P_RING_coarse_slider,
    'NPDPII_THR_S_P_RING_fine_slider': NPDPII_THR_S_P_RING_fine_slider,
    'PS_WEIGHT_EXC_S_N_INHPOP_coarse_slider': PS_WEIGHT_EXC_S_N_INHPOP_coarse_slider,
    'PS_WEIGHT_EXC_S_N_INHPOP_fine_slider': PS_WEIGHT_EXC_S_N_INHPOP_fine_slider,
    'NPDPIE_THR_S_P_INHPOP_coarse_slider': NPDPIE_THR_S_P_INHPOP_coarse_slider,
    'NPDPIE_THR_S_P_INHPOP_fine_slider': NPDPIE_THR_S_P_INHPOP_fine_slider,
    'I0_slider': I0_slider,
    'run_dynapse_button': run_dynapse_button
}


In [ ]:
def interactive_dynapse(run_dynapse_button,
                          num_neurons_slider,
                          stimulus_center_slider,
                          stimulus_width_slider,
                          velocity_input_slider,
                          duration_box,
                          input_duration_box,
                          I0_slider):
    
    global sink_node
    global NBINS
    global ring_pops
    
    if not run_dynapse_button:
        clear_output(wait=False)
        return
    
    # ----------------  stimulus parameters ----------------
    n_pts = 1000                  # number of samples
    t_end = 1.0                   # seconds  (→ dt = 1 ms)
    t = np.linspace(0, t_end, n_pts, endpoint=False)
    dt = t_end / n_pts            # simulation time-step (s)

    # Define positions of neurons in a ring - use the slider value
    num_neurons = num_neurons_slider
    positions = np.linspace(0, 2*np.pi, num_neurons, endpoint=False)

    # Create stimulus parameters using slider values
    I0 = I0_slider                # peak current amplitude from slider

    # Calculate input currents based on positions and stimulus parameters from sliders
    d = np.angle(np.exp(1j * (positions - stimulus_center_slider)))
    I_ext_array = I0 * np.exp(-(d**2) / (2 * stimulus_width_slider**2))

    # ----------------  LIF neuron parameters ----------------------
    tau_m = 20e-3                 # 20 ms membrane time constant
    R_m = 100e6                   # 100 MΩ  (=> C = tau/R)
    C_m = tau_m / R_m
    v_rest = -65e-3               # -65 mV
    v_reset = -65e-3
    v_thresh = -50e-3             # spike threshold
    t_ref = 2e-3 
    
    
    # Clear any previous figures
    plt.close('all')

    # ----------------  simulation loop for multiple neurons ----------------------------
    v = np.full(num_neurons, v_rest)  # initialize all neurons at rest
    next_ok = np.zeros(num_neurons)   # refractory period end time for each neuron
    v_trace = np.empty((num_neurons, n_pts))
    spikes = [[] for _ in range(num_neurons)]  # list of spike times for each neuron
    spike_times_all = [] # list of all spike times for all neurons
    spike_ids_all = [] # list of all spikegen ids for all neurons

    for k in range(n_pts):
        for i in range(num_neurons):
            if t[k] >= next_ok[i]:          # not in refractory
                dv = (-(v[i] - v_rest) + R_m * I_ext_array[i]) / (R_m * C_m) * dt
                v[i] += dv
                if v[i] >= v_thresh:        # spike!
                    spikes[i].append(t[k])
                    spike_times_all.append(t[k])
                    spike_ids_all.append(i)  
                    v[i] = v_reset
                    next_ok[i] = t[k] + t_ref
            v_trace[i, k] = v[i]


            # Sort spike_times_all and reorder spike_ids_all2 accordingly
            if len(spike_times_all) > 0:  # Check if there are any spikes
                # Get the indices that would sort spike_times_all2
                sort_indices = np.argsort(spike_times_all)
                
                # Apply the sorting to both arrays
                spike_times_all = [spike_times_all[i] for i in sort_indices]
                spike_ids_all = [spike_ids_all[i] for i in sort_indices]

    spike_times_all = np.array(spike_times_all)  # Convert to numpy array for dynapse interface

    plot_figures_flag = False  # Set to True to plot figures
    if plot_figures_flag:
        # ----------------  plots --------------------------------------
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=False)

        # Plot input currents for each neuron
        ax1.plot(positions*180/np.pi, I_ext_array*1e12, 'o-')
        ax1.set_xlabel('Position (degrees)')
        ax1.set_ylabel('Current (pA)')
        ax1.set_title('Input current by position')
        ax1.grid(True)
        # Set x-axis to display 0-360 degree range with appropriate ticks
        ax1.set_xlim(0, 360)
        ax1.set_xticks(np.arange(0, 361, 360/num_neurons))  # 45 degree increments
        # Convert neuron positions to degrees for scatter points
        neuron_positions_deg = positions * 180/np.pi
        # Add markers at actual neuron positions
        ax1.scatter(neuron_positions_deg, I_ext_array*1e12, color='red', s=50, zorder=3)

        # Plot raster plot of spikes
        for i in range(num_neurons):
            if len(spikes[i]) > 0:
                ax2.plot(np.array(spikes[i]), np.full_like(spikes[i], i), '|', markersize=10, color='red')
        ax2.set_xlabel('Time (s)')
        ax2.set_xlim(0, t_end)
        ax2.set_ylabel('Neuron index')
        ax2.set_title('Spike raster plot')
        ax2.set_ylim(-0.5, num_neurons-0.5)
        ax2.set_yticks(range(num_neurons))
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    import params_all_cores
    import importlib
    importlib.reload(params_all_cores)

    # initiate network 
    net_gen = NetworkGenerator()
    net_gen.clear_network()

    # create VIRTUAL spikegens, one per ring attractor neural pop 
    spikegen_ids = [(0, 0, n) for n in range(10)]

    spikegens = []
    for spikegen_id in spikegen_ids:
        spikegens.append(Neuron(spikegen_id[0], spikegen_id[1],spikegen_id[2], True))
        
        
    # create neuron populations in the ring
    chip = 0 # chip 0 for the ring attractor neurons
    core = 1 # core 1 for excitatory neurons
    npop = 3 # number of neurons in each population
    NBINS = 10 # number of populations in the ring

    start_population_index = 20 # starting index for the first population in core 0

    # Create neuron populations for each band in core 0
    ring_pops = [
        [Neuron(chip, core, j) for j in range(start_population_index + (i * npop), start_population_index + ((i + 1) * npop))]
        for i in range(NBINS)
    ]

    # create inhibitory population that connects to all other pops
    core_inh = 2 # core 2 for inhibitory neurons
    start_inh_neuron = 4 # starting index for the first inhibitory neuron in core 2
    npop_inh = 6 # number of inhibitory neurons
    pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]

    # connect the first spikegen to all the neurons of the fourth pop in the ring
    for k in range(len(ring_pops)):
        for i in ring_pops[k]:
            #if i%2 == 0: # only connect to even neurons in the population
            net_gen.add_connection(spikegens[k], i, dyn1.Dynapse1SynType.AMPA)

    # Use the autapse button value to determine if self-connections are allowed
    if autapse_button:
        p_E_E = 0.5
        # self excitation in each neural population in the ring
        for pop in ring_pops:
            for pre in pop:
                for post in pop:
                    if pre is not post and np.random.rand() < p_E_E:
                        net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                        
    # ******** MEXICAN HAT CONNECTIONS **********

    #excitatory connections to first neighbors
    OFFSET_1 = (-1, 1)         
    for i, pop_i in enumerate(ring_pops):
        for pre in pop_i:
            for d in OFFSET_1:
                j = (i + d) % NBINS          # wrap around
                for post in ring_pops[j]:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                    #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)


    OFFSET_2 = (-2, 2)          # ±3 bins wide "hat"
    # excitatory connections to second neighbors
    for i, pop_i in enumerate(ring_pops):
        for pre in pop_i:
            for d in OFFSET_2:
                j = (i + d) % NBINS          # wrap around
                for post in ring_pops[j]:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                    #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)

    OFFSET_3 = (-3, 3)          # ±3 bins wide "hat"

    # excitatory connections to third neighbors
    for i, pop_i in enumerate(ring_pops):
        for pre in pop_i:
            for d in OFFSET_3:
                j = (i + d) % NBINS          # wrap around
                for post in ring_pops[j]:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

    # todo inhibitory connections to all of the other pops
    OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited
    for i, pop_i in enumerate(ring_pops):
        for pre in pop_i:
            for d in OFFSET_inh:
                j = (i + d) % NBINS          # wrap around
                for post in ring_pops[j]:
                    net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

    # INH → EXC  (global inhibition pop to all pops in the ring)
    for inh in pop_inhibitory:
        for pop in ring_pops:
            for exc in pop:
                net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

    # EXC → INH  (drive the global inhibition pop from all pops in the ring)
    for pop in ring_pops:
        for exc in pop:
            for inh in pop_inhibitory:
                net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)
    
    # Print the network for verification (optional)
    # print(net_gen.network)

    # make a dynapse1config using the network
    new_config = net_gen.make_dynapse1_configuration()

    # apply the configuration
    model.apply_configuration(new_config)

    # Set hardware parameters using custom parameters
    from params_all_cores import set_params
    set_params(model)
        
    # Use the duration values from input boxes    
    D_stim = input_duration_box.value # stimulus duration (s)
    D_post_stim = duration_box.value - input_duration_box.value # post stimulus duration (s)
    D = duration_box.value # total duration (s)

    fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 

    # get events of selected neurons
    monitored_neurons = [
        (n.chip_id, n.core_id, n.neuron_id)   
        for pop in ring_pops
        for n in pop
    ]

    monitored_neurons.extend([
        (neuron.chip_id, neuron.core_id, neuron.neuron_id)
        for neuron in pop_inhibitory  
    ])

    graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
    graph.start()

    # clear the buffer
    sink_node.get_events()

    # select the neurons to monitor
    # print("monitored neurons:", monitored_neurons)
    filter_node.set_neurons(monitored_neurons)

    api.reset_timestamp()

    # print("spike times all, ", spike_times_all)
    # print("spike_ids_all, ", spike_ids_all)

    ut.set_fpga_spike_gen(
        fpga_spike_gen,
        spike_times_all,
        spike_ids_all,
        target_chips=[0] * len(spike_ids_all),
        isi_base=900,
        repeat_mode=False)

    fpga_spike_gen.start()

    #time.sleep(D)

    #fpga_spike_gen.stop()

    # graph.stop()

In [ ]:

# Initialize result_fig to None
result_fig = None

# Function to update the results display
def update_results_display(results_output):
    global result_fig
    with results_output:
        results_output.clear_output(wait=True)
        try:
            if result_fig is not None:
                display(result_fig)
            else:
                print("Run a simulation to see results")
        except Exception as e:
            print(f"Error displaying figure: {e}")

# Main simulation parameters
sim_params = VBox([
    HTML(value="<h3>Simulation Parameters</h3>"),
    input_duration_box,
], layout=Layout(border='1px solid lightgray', padding='10px', margin='5px'))

# Input parameters
input_params = VBox([
    HTML(value="<h3>Input Parameters</h3>"),
    I0_slider,
    stimulus_center_slider,
    stimulus_width_slider,
    velocity_input_slider
], layout=Layout(border='1px solid lightgray', padding='10px', margin='5px'))


# Button container
button_box = HBox([
    run_dynapse_button
], layout=Layout(justify_content='center', margin='10px', padding='10px'))

# Create the main layout with tabs
# Tab 1: Basic parameters and input
basic_tab = VBox([
    HBox([sim_params, input_params], layout=Layout(justify_content='center')),
    button_box
], layout=Layout(padding='10px'))


out = interactive_output(interactive_dynapse, widgets)

# # Results tab
# results_tab = VBox([
#     HTML(value="<h3>Simulation Results</h3>"),
#     results_output
# ], layout=Layout(padding='10px'))

# # Initialize the display with an empty results figure
# update_results_display(results_output)

# Create and configure the tab widget with three tabs (including results)
tabs = Tab()
tabs.children = [basic_tab]
tabs.set_title(0, 'Basic Parameters')

# Display the dashboard
display(tabs)


In [16]:
spike_thread = threading.Thread(target=collect_spikes, args=sink_node)
spike_thread.daemon = True
spike_thread.start()

Exception in thread Thread-10 (collect_spikes):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/home/bmaacaron-iit.local/.virtualenvs/DynapseRA/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
TypeError: __main__.collect_spikes() argument after * must be an iterable, not int


In [ ]:
# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))

# Setup raster plot in first subplot
scatter = ax_raster.scatter([], [], s=10, alpha=0.6)
xlim_max = 10
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
# Create positions for neurons (0 to 2π for the ring)
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)
rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0  # seconds - time window for calculating rates
last_update_time = 0

# Real-time plotting loop
try:
    if len(eventsBuffer) > 0:
        # Extract spike data for raster plot
        spikesID = [e.neuron_id for e in eventsBuffer]
        spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]  # Convert to seconds
        
        print(spikesTimes)
        print(spikesID)
        
        if not spikesID:
            pass
            
        # Update raster plot
        ax_raster.set_ylim(min(spikesID) - 0.5, max(spikesID) + 0.5)
        scatter.set_offsets(np.column_stack((spikesTimes, spikesID)))
        
        # Update time window
        current_time = max(spikesTimes)
        ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
        
        # Calculate firing rates across the ring (using recent time window)
        window_start = current_time - window_size
        recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
        
        # Count spikes for each bin in the ring
        firing_rates = np.zeros(NBINS)
        
        # Map neuron IDs to their bin/position in the ring
        for event in recent_events:
            neuron_id = event.neuron_id
            for i, pop in enumerate(ring_pops):
                pop_ids = [n.neuron_id for n in pop]
                if neuron_id in pop_ids:
                    firing_rates[i] += 1
                    break
        
        # Convert to Hz (spikes per second)
        firing_rates = firing_rates / window_size
        
        # Update firing rate plot
        rate_line.set_ydata(firing_rates)
        max_rate = max(firing_rates) if any(firing_rates > 0) else 10
        ax_rate.set_ylim(0, max_rate * 1.2)  # Add 20% margin
        
        # Refresh both plots
        fig.canvas.draw_idle()
        plt.pause(0.0001)  # Shorter pause for smoother updates
        
except Exception as e:
    print(f"Plotting error: {e}")
    import traceback
    traceback.print_exc()  # Print detailed error information